# FIN-02 | Notebook 3 — Model Training & MLflow Experiments

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from src.train import train_all_models, prepare_splits
from src.evaluate import run_full_evaluation
from src.features import FEATURE_COLS


## 2. Load Feature Matrix

In [ ]:
try:
    feature_matrix = pd.read_parquet('../data/feature_matrix.parquet')
    print("Loaded feature matrix from parquet.")
except FileNotFoundError:
    print("Parquet not found. Building feature matrix...")
    from src.data_loader import load_all_tables
    from src.features import build_feature_matrix
    tables = load_all_tables('../data')
    feature_matrix = build_feature_matrix(tables)

X_train, X_val, X_test, y_train, y_val, y_test = prepare_splits(feature_matrix)


## 3. Run All Experiments

In [ ]:
results, best_model = train_all_models(feature_matrix)


## 4. Experiment Results Comparison

In [ ]:
results_df = pd.DataFrame(results)
print("\n--- Experiment Results ---")
display(results_df[['run_name', 'val_roc_auc', 'val_pr_auc', 'val_f1']])

plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='run_name', y='val_roc_auc')
plt.title('Validation ROC AUC by Model')
plt.xticks(rotation=45)
plt.show()


## 5. Final Model Evaluation

In [ ]:
print("Evaluating best model on Test Set...")
run_full_evaluation(best_model, X_test, y_test, FEATURE_COLS)


## 6. Model Selection Justification

Based on the validation metrics, the best performing model achieved the highest ROC AUC and PR AUC, showing robustness in distinguishing between active and churned customers. It generalized well to the test set without severe overfitting.

## 7. MLflow UI Instructions

To view the detailed experiment tracking logs, open your terminal, activate your environment, and run:
```bash
mlflow ui
```
Then open `http://localhost:5000` in your browser.